In [1]:
!pip install transformers torch --quiet

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")
print(f"PyTorch version : {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on      : {device.upper()}")

Libraries imported successfully!
PyTorch version : 2.10.0+cpu
Running on      : CPU


In [3]:
MODEL_NAME = "microsoft/DialoGPT-medium"
print("Loading model, please wait...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()
print("Model loaded!")

Loading model, please wait...


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded!


In [6]:
def generate_response(user_input, chat_history_ids=None):

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.75,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )

    response_ids = output_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    return response_text, output_ids

In [7]:
def run_chatbot():
    print("=" * 55)
    print("        DialoGPT Chatbot - Hugging Face")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("(type 'exit' or 'quit' to stop)\n")
    chat_history_ids = None
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day!")
            break
        if not user_input:
            print("Chatbot: Please say something!")
            continue
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        if not response.strip():
            response = "Could you rephrase that? I didn't quite get it."
        print(f"Chatbot: {response}\n")
        # trim history to avoid hitting the token limit
        if chat_history_ids is not None and chat_history_ids.shape[-1] > 800:
            chat_history_ids = chat_history_ids[:, -800:]
run_chatbot()

        DialoGPT Chatbot - Hugging Face
Chatbot: Hello! I am your AI assistant. How can I help you today?
(type 'exit' or 'quit' to stop)

You: Hello
Chatbot: Hey! : 3

You: What is artificial intelligence? 
Chatbot: I don't know, you tell me.

You: Who created python?
Chatbot: The brain's creator I think...

You: What is machine learning?
Chatbot: It makes things more complex and more accurate. That's what a lot of these programs do.

You: Thank you
Chatbot: No problemo, welcome to the club!

You: exit
Chatbot: Goodbye! Have a great day!


In [8]:
def demo_conversation(inputs):
    print("=" * 55)
    print("        DialoGPT Chatbot - Demo")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")
    chat_history_ids = None
    for user_input in inputs:
        print(f"You    : {user_input}")
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye!")
            break
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        if not response.strip():
            response = "Could you rephrase that? I didn't quite get it."
        print(f"Chatbot: {response}\n")
        if chat_history_ids is not None and chat_history_ids.shape[-1] > 800:
            chat_history_ids = chat_history_ids[:, -800:]
sample_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What is machine learning?",
    "Thank you",
    "exit"
]
demo_conversation(sample_inputs)

        DialoGPT Chatbot - Demo
Chatbot: Hello! I am your AI assistant. How can I help you today?

You    : Hello
Chatbot: Hey! I've met you, on the other thread...

You    : What is Artificial Intelligence?
Chatbot: If it's what we think. It's more like a virus that spreads through our brains and minds, but it doesn't exist until we get to age 19 or so..? :D

You    : Who created Python?
Chatbot: I did not create python. python was an accident when i wanted something as simple as web development

You    : What is machine learning?
Chatbot: The computer programs?

You    : Thank you
Chatbot: It wasn t me, but yes

You    : exit
Chatbot: Goodbye!
